In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [2]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})
        page = tif.pages[0]

        def _rational(name):
            tag = page.tags.get(name)
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = _rational("XResolution")
        yres = _rational("YResolution")
        resunit_tag = page.tags.get("ResolutionUnit")
        resunit = int(resunit_tag.value) if resunit_tag is not None else None

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta


def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata."""
    ij = meta["imagej"]
    md = {"axes": meta["axes"]}
    for key in ("spacing", "unit", "finterval", "fps", "hyperstack",
                "mode", "channels", "slices", "frames"):
        if key in ij:
            md[key] = ij[key]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    tifffile.imwrite(path, arr, **kwargs)

In [3]:
# Curate the segmentation channel (c2) of each tif. A single napari viewer is
# open at any moment, showing all three channels (BF, fluo, segmentation).
# Edit the mask, then click "Save & Next" (or close the window). The viewer
# then reopens with the next image automatically.
#
# Two curation modes are supported:
#   - mode="middle": only the middle z-slice is shown/edited in 2D. The curated
#     middle-slice mask is duplicated across every z-slice to form a prism, and
#     the result is saved as a NEW tif in `output_folder`. BF (c0) and fluo (c1)
#     are kept unchanged.
#   - mode="stack": the full 3D stack is shown/edited (napari z-slider). The
#     curated 3D mask is written back to the segmentation channel and saved IN
#     PLACE (overwriting the original file). BF (c0) and fluo (c1) are unchanged.
#
# Pixel-size / spacing metadata is preserved either way.
#
# In a notebook the Qt event loop is already running, so we can't use a
# blocking `for` loop (that opens every viewer at once). Instead we chain
# images together through the viewer's close event.

class MaskCurator:
    def __init__(self, images, mode="middle", start_index=0, brightfield_channel=0, fluorescence_channel=1, mask_channel=2, output_folder=None):
        if mode not in ("middle", "stack"):
            raise ValueError("mode must be 'middle' or 'stack'")
        self.images = images
        self.mode = mode
        self.index = start_index
        self.viewer = None
        self.arr = None
        self.meta = None
        self.labels_layer = None
        self.mid_z = None
        self._orig_close = None
        self._advancing = False
        self.brightfield_channel = brightfield_channel
        self.fluorescence_channel = fluorescence_channel
        self.mask_channel = mask_channel
        self.output_folder = output_folder

    def start(self):
        self._open_current()

    def _open_current(self):
        if self.index >= len(self.images):
            print("All images curated.")
            return

        path = self.images[self.index]
        print(f"[{self.index + 1}/{len(self.images)}] Curating {path.name}")

        self.arr, self.meta = read_image_and_meta(path)
        n_z = self.arr.shape[0]
        self.mid_z = n_z // 2

        if self.mode == "middle":
            # Middle slice of each channel (2D Y, X).
            brightfield = self.arr[self.mid_z, self.brightfield_channel, :, :]
            fluorescence = self.arr[self.mid_z, self.fluorescence_channel, :, :]
            mask = self.arr[self.mid_z, self.mask_channel, :, :]
            title = (f"[{self.index + 1}/{len(self.images)}] {path.name} "
                     f"(z={self.mid_z})")
        else:
            # Full 3D stack of each channel (3D Z, Y, X).
            brightfield = self.arr[:, self.brightfield_channel, :, :]
            fluorescence = self.arr[:, self.fluorescence_channel, :, :]
            mask = self.arr[:, self.mask_channel, :, :]
            title = (f"[{self.index + 1}/{len(self.images)}] {path.name} "
                     f"(3D stack)")

        self.viewer = napari.Viewer(title=title)
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive")
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive")
        self.labels_layer = self.viewer.add_labels(
            mask.astype(np.int32), name="mask"
        )

        next_btn = PushButton(text="Save & Next")
        next_btn.clicked.connect(self.viewer.close)
        self.viewer.window.add_dock_widget(next_btn, area="right",
                                           name="curation")

        # Route the window close (button or X) through our save+advance logic.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    def _on_close(self, event):
        if not self._advancing:
            self._advancing = True
            path = self.images[self.index]
            curated = self.labels_layer.data.astype(self.arr.dtype)
            out = self.arr.copy()

            if self.mode == "middle":
                # Curated 2D middle slice -> duplicate across every z (prism).
                n_z = self.arr.shape[0]
                out[:, self.mask_channel, :, :] = np.broadcast_to(
                    curated, (n_z,) + curated.shape
                )
                # Save as a NEW tif in the output folder.
                out_path = self.output_folder / path.name
            else:
                # Curated full 3D mask -> write back as segmentation channel.
                out[:, self.mask_channel, :, :] = curated
                # Save IN PLACE, overwriting the original file.
                out_path = path
            # BF (c0) and fluo (c1) remain unchanged in `out`.

            save_image(out_path, out, self.meta)
            print(f"    saved -> {out_path}")

            self.index += 1
            # Open the next image once this window has finished closing.
            QTimer.singleShot(200, self._open_next)
        self._orig_close(event)

    def _open_next(self):
        self._advancing = False
        self._open_current()

In [5]:
vascumap_masks_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\masks_to_curate")
expanded_middle_slice_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks")
expanded_middle_slice_folder.mkdir(parents=True, exist_ok=True)

images = sorted(vascumap_masks_folder.glob("*.tif"))
images = images[10:]
print(f"{len(images)} images found")

58 images found


## Option 1 — Curate the middle slice, save to output folder

Opens the **middle z-slice** of each image in 2D. The curated middle-slice
mask is expanded across the full z-stack (prism) and saved as a **new tif** in
`output_folder`. Brightfield and fluorescence channels are left unchanged.

In [ ]:
# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(images, mode="middle", start_index=0)
curator.start()

## Option 2 — Curate the full 3D stack, save in place

Opens the **full 3D stack** of each image (use the napari z-slider to move
through slices). The curated 3D mask is written back to the segmentation
channel and saved **in place**, overwriting the original file. Brightfield and
fluorescence channels are left unchanged.

In [ ]:
# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(images = sorted(expanded_middle_slice_folder.glob("*.tif")), mode="stack", start_index=0)
curator.start()

[1/9] Curating 20260514_FL37_ARi_device1_infocus.tif


    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device1_infocus.tif
[2/9] Curating 20260514_FL37_ARi_device4_infocus.tif
    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device4_infocus.tif
[3/9] Curating 20260514_FL37_UTD_device2_infocus.tif
    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_UTD_device2_infocus.tif
[4/9] Curating 20260514_FL37_UTD_device4_infocus.tif
